Code fine-tune Model 3


## Bước 1: Nạp dữ liệu và kiểm tra môi trường

In [1]:
# Path giúp tạo đường dẫn tệp theo hệ điều hành đang dùng.
from pathlib import Path

# json dùng để đọc từng dòng trong file JSONL.
import json

# PyTorch dùng ở các bước sau để huấn luyện PhoBERT.
import torch

# Lấy thư mục hiện tại mà notebook đang chạy.
project_root = Path.cwd()

# Nếu notebook chạy từ notebooks/, quay lên thư mục gốc dự án.
if project_root.name.lower() == "notebooks":
    project_root = project_root.parent

# Tạo đường dẫn tới hai tập dữ liệu đã chốt.
train_path = project_root / "data" / "processed" / "train.json"
dev_path = project_root / "data" / "processed" / "dev.json"

# Hàm đọc file JSON Lines: mỗi dòng là một mẫu dữ liệu riêng.
def load_jsonl(file_path):
    examples = []

    # Mở file bằng UTF-8 để đọc đúng nội dung tiếng Việt.
    with file_path.open("r", encoding="utf-8") as file:
        for line in file:
            # Bỏ qua dòng trống, nếu có.
            if line.strip():
                examples.append(json.loads(line))

    return examples

# Nạp dữ liệu train và dev vào hai danh sách riêng.
train_examples = load_jsonl(train_path)
dev_examples = load_jsonl(dev_path)

# Đếm số câu có đáp án và không có đáp án trong tập train.
train_no_answer = sum(
    example["is_impossible"] for example in train_examples
)
train_has_answer = len(train_examples) - train_no_answer

# Đếm hai nhóm tương tự trong tập dev.
dev_no_answer = sum(
    example["is_impossible"] for example in dev_examples
)
dev_has_answer = len(dev_examples) - dev_no_answer

# In số mẫu để chắc chắn đã nạp đúng dữ liệu.
print("Train:", len(train_examples))
print("  Có đáp án:", train_has_answer)
print("  Không có đáp án:", train_no_answer)

print("Dev:", len(dev_examples))
print("  Có đáp án:", dev_has_answer)
print("  Không có đáp án:", dev_no_answer)

# Kiểm tra notebook đang thấy GPU hay không.
print("CUDA sẵn sàng:", torch.cuda.is_available())

# Nếu có GPU, in tên GPU đang dùng.
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Train: 8000
  Có đáp án: 5417
  Không có đáp án: 2583
Dev: 3814
  Có đáp án: 2653
  Không có đáp án: 1161
CUDA sẵn sàng: True
GPU: NVIDIA GeForce RTX 5070 Ti


Mục đích của bước này là xác nhận notebook đọc đúng dữ liệu trước khi huấn luyện. Kết quả mong đợi: train 8.000 mẫu, dev 3.814 mẫu, và máy nhận đúng gpu

-----------------------------------------------------------------------------------------

## Bước 2: Thử tách từ và kiểm tra đáp án
Notebook 06 hiện mới có tiêu đề. Trước khi huấn luyện PhoBERT, ta cần xem underthesea tách câu hỏi/context ra sao và vị trí đáp án thay đổi thế nào. word_tokenize(..., format="text") nối các âm tiết cùng một từ bằng dấu gạch dưới, ví dụ Chàng trai thành Chàng_trai

In [2]:
%pip install underthesea

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
# word_tokenize dùng underthesea để tách từ tiếng Việt.
from underthesea import word_tokenize

# Chọn một mẫu có đáp án để kiểm tra phép tách từ.
example = next(
    item for item in train_examples
    if not item["is_impossible"]
    and item["answers"]["text"]
    and item["answers"]["answer_start"]
)

# Lấy context, câu hỏi, vị trí bắt đầu và nội dung đáp án gốc.
question = example["question"]
context = example["context"]
answer_text = example["answers"]["text"][0]
answer_start = example["answers"]["answer_start"][0]
answer_end = answer_start + len(answer_text)

# Cắt đáp án ra khỏi context gốc bằng vị trí ký tự ban đầu.
answer_from_context = context[answer_start:answer_end]

# Tách từ; format="text" nối các phần của một từ bằng dấu gạch dưới.
question_segmented = word_tokenize(question, format="text")
context_segmented = word_tokenize(context, format="text")
answer_segmented = word_tokenize(answer_text, format="text")

# Tìm đáp án đã tách trong context đã tách.
segmented_answer_start = context_segmented.find(answer_segmented)

print("Câu hỏi gốc:", question)
print("Câu hỏi sau tách từ:", question_segmented)
print()
print("Đáp án gốc:", answer_text)
print("Đoạn cắt theo answer_start:", answer_from_context)
print("Vị trí đáp án trong context gốc:", answer_start)
print()
print("Đáp án sau tách từ:", answer_segmented)
print("Vị trí đáp án trong context đã tách:", segmented_answer_start)
print("Tìm thấy đáp án đã tách:", segmented_answer_start >= 0)

# Kiểm tra answer_start ban đầu có trỏ đúng đáp án trong context gốc không.
print("answer_start khớp context gốc:", answer_from_context == answer_text)

c:\Users\havuh\OneDrive\Desktop\HocTap\Vietnamese_Extractive_Question_Answering_on_UIT_ViQuAD_2.0_Project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Câu hỏi gốc: Birmingham phát triển nổi bật về lĩnh vực gì?
Câu hỏi sau tách từ: Birmingham phát_triển nổi_bật về lĩnh_vực gì ?

Đáp án gốc: hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu
Đoạn cắt theo answer_start: hoạt động văn chương, âm nhạc, nghệ thuật và sân khấu
Vị trí đáp án trong context gốc: 179

Đáp án sau tách từ: hoạt_động văn_chương , âm_nhạc , nghệ_thuật và sân_khấu
Vị trí đáp án trong context đã tách: 181
Tìm thấy đáp án đã tách: True
answer_start khớp context gốc: True


Điểm cần quan sát: answer_start là vị trí trong context gốc. Sau khi tách từ, độ dài và vị trí ký tự thay đổi, nên không được dùng nguyên answer_start cũ trên context_segmented. Bước này chỉ giúp ta nhìn rõ thay đổi đó; chưa huấn luyện model.

## Bước 3: Ánh xạ đáp án sau khi tách từ cho toàn bộ dữ liệu

PhoBERT cần câu hỏi và context đã tách từ. Dấu cách bên trong một từ có thể đổi thành dấu gạch dưới, vì vậy answer_start gốc không thể dùng trực tiếp trên context mới. Ta căn chỉnh theo thứ tự ký tự sau khi bỏ khoảng trắng và dấu gạch dưới.

Nếu answer_start gốc sai, chỉ tự sửa khi đáp án có đúng một vị trí trong context. Trường hợp còn mơ hồ sẽ được báo lỗi; không âm thầm bỏ mẫu.


In [4]:
from difflib import SequenceMatcher


def bo_dau_phan_tach(text):
    # Bỏ khoảng trắng và dấu gạch dưới do tách từ thêm vào.
    # Lưu vị trí ký tự để đổi ngược về chuỗi ban đầu.
    characters = []
    positions = []

    for index, character in enumerate(text):
        if character not in (" ", chr(9), chr(10), chr(13), "_"):
            characters.append(character)
            positions.append(index)

    return "".join(characters), positions


def tim_tat_ca_vi_tri(text, phrase):
    # Tìm tất cả vị trí có thể có cùng chuỗi đáp án.
    positions = []
    search_from = 0

    while True:
        position = text.find(phrase, search_from)
        if position == -1:
            break
        positions.append(position)
        search_from = position + 1

    return positions


def anh_xa_span(context, answer_text, answer_start):
    # Chỉ sửa answer_start nếu đáp án gốc xuất hiện đúng một lần.
    answer_end = answer_start + len(answer_text)
    offset_repaired = False

    if context[answer_start:answer_end] != answer_text:
        possible_positions = tim_tat_ca_vi_tri(context, answer_text)
        if len(possible_positions) != 1:
            return None, None, None, False, "offset gốc sai hoặc đáp án bị lặp"
        answer_start = possible_positions[0]
        offset_repaired = True

    # Tách cả context một lần để cách tách từ nhất quán.
    context_segmented = word_tokenize(context, format="text")
    original_compact, _ = bo_dau_phan_tach(context)
    segmented_compact, segmented_positions = bo_dau_phan_tach(context_segmented)

    # Tính span của đáp án trong context gốc sau khi bỏ dấu phân tách.
    answer_before, _ = bo_dau_phan_tach(context[:answer_start])
    answer_compact, _ = bo_dau_phan_tach(answer_text)
    compact_start = len(answer_before)
    compact_end = compact_start + len(answer_compact)

    if original_compact[compact_start:compact_end] != answer_compact:
        return None, None, None, offset_repaired, "đáp án không khớp vị trí gốc"
    if not answer_compact or compact_end > len(original_compact):
        return None, None, None, offset_repaired, "span rỗng hoặc nằm ngoài context"

    # Ghép các đoạn ký tự còn giống nhau; không yêu cầu cả context phải giống hệt.
    # Ví dụ: ký tự ở phần khác của context bị đổi nhưng span đáp án vẫn còn nguyên.
    source_to_segmented = [None] * len(original_compact)
    matcher = SequenceMatcher(None, original_compact, segmented_compact)

    for matching_block in matcher.get_matching_blocks():
        for offset in range(matching_block.size):
            source_index = matching_block.a + offset
            segmented_index = matching_block.b + offset
            source_to_segmented[source_index] = segmented_index

    mapped_positions = source_to_segmented[compact_start:compact_end]
    aligned = (
        len(mapped_positions) == len(answer_compact)
        and all(position is not None for position in mapped_positions)
        and mapped_positions == list(
            range(mapped_positions[0], mapped_positions[0] + len(answer_compact))
        )
    )

    # Nếu căn chỉnh toàn context không xác định được span,
    # chỉ dùng vị trí khi đáp án xuất hiện duy nhất trong context sau tách.
    # Nếu có nhiều vị trí thì không đoán, mẫu sẽ báo lỗi an toàn.
    if aligned:
        segmented_compact_start = mapped_positions[0]
    else:
        answer_occurrences = tim_tat_ca_vi_tri(segmented_compact, answer_compact)
        if len(answer_occurrences) != 1:
            return None, None, None, offset_repaired, "không xác định duy nhất span đáp án sau tách"
        segmented_compact_start = answer_occurrences[0]

    segmented_compact_end = segmented_compact_start + len(answer_compact)
    if segmented_compact[segmented_compact_start:segmented_compact_end] != answer_compact:
        return None, None, None, offset_repaired, "ký tự trong span đáp án bị thay đổi"
    if segmented_compact_end > len(segmented_positions):
        return None, None, None, offset_repaired, "span sau tách nằm ngoài context"

    # Đổi vị trí rút gọn về vị trí ký tự trong context đã tách.
    mapped_start = segmented_positions[segmented_compact_start]
    mapped_end = segmented_positions[segmented_compact_end - 1] + 1
    mapped_answer = context_segmented[mapped_start:mapped_end]
    mapped_compact, _ = bo_dau_phan_tach(mapped_answer)

    if mapped_compact != answer_compact:
        return None, None, None, offset_repaired, "span mới không khớp đáp án gốc"

    return context_segmented, mapped_answer, mapped_start, offset_repaired, None


def xu_ly_tap(examples):
    processed = []
    errors = []
    repaired_count = 0

    for example in examples:
        new_example = example.copy()
        new_example["question"] = word_tokenize(example["question"], format="text")

        if example["is_impossible"]:
            # Câu hỏi không có đáp án không cần span để huấn luyện.
            new_example["context"] = word_tokenize(example["context"], format="text")
            processed.append(new_example)
            continue

        answers = example.get("answers", {})
        texts = answers.get("text", [])
        starts = answers.get("answer_start", [])

        if not texts or not starts:
            errors.append({"id": example.get("id"), "reason": "thiếu answer_text hoặc answer_start"})
            processed.append(example.copy())
            continue

        context_new, answer_new, start_new, repaired, reason = anh_xa_span(
            example["context"], texts[0], starts[0]
        )
        if reason:
            errors.append({"id": example.get("id"), "reason": reason})
            processed.append(example.copy())
            continue

        if repaired:
            repaired_count += 1
        new_example["context"] = context_new
        new_example["answers"] = {"text": [answer_new], "answer_start": [start_new]}
        processed.append(new_example)

    return processed, errors, repaired_count


train_segmented, train_errors, train_repaired = xu_ly_tap(train_examples)
dev_segmented, dev_errors, dev_repaired = xu_ly_tap(dev_examples)

print("Train:", len(train_segmented), "mẫu; lỗi:", len(train_errors), "; offset tự sửa:", train_repaired)
print("Dev:", len(dev_segmented), "mẫu; lỗi:", len(dev_errors), "; offset tự sửa:", dev_repaired)
print("Ví dụ lỗi train:", train_errors[:5])
print("Ví dụ lỗi dev:", dev_errors[:5])

if train_errors or dev_errors:
    from collections import Counter

    train_reason_counts = Counter(
        error["reason"] for error in train_errors
    )
    dev_reason_counts = Counter(
        error["reason"] for error in dev_errors
    )

    # Show the cause and example IDs directly in the exception traceback.
    error_message = (
        "\u00c1nh xạ chưa hoàn tất. "
        f"Train: {len(train_errors)} lỗi, theo lý do {dict(train_reason_counts)}. "
        f"Dev: {len(dev_errors)} lỗi, theo lý do {dict(dev_reason_counts)}. "
        f"Mẫu train: {train_errors[:10]}. "
        f"Mẫu dev: {dev_errors[:10]}"
    )
    raise ValueError(error_message)


Train: 8000 mẫu; lỗi: 74 ; offset tự sửa: 45
Dev: 3814 mẫu; lỗi: 29 ; offset tự sửa: 13
Ví dụ lỗi train: [{'id': '0126-0020-0004', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0033-0047-0004', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0021-0037-0004', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0109-0003-0004', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0126-0039-0003', 'reason': 'không xác định duy nhất span đáp án sau tách'}]
Ví dụ lỗi dev: [{'id': '0002-0013-0005', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0004-0009-0002', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0004-0009-0003', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0004-0028-0001', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0004-0032-0010', 'reason': 'không xác định duy nhất span đáp án sau tách'}]


ValueError: Ánh xạ chưa hoàn tất. Train: 74 lỗi, theo lý do {'không xác định duy nhất span đáp án sau tách': 71, 'offset gốc sai hoặc đáp án bị lặp': 3}. Dev: 29 lỗi, theo lý do {'không xác định duy nhất span đáp án sau tách': 29}. Mẫu train: [{'id': '0126-0020-0004', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0033-0047-0004', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0021-0037-0004', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0109-0003-0004', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0126-0039-0003', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0010-0019-0002', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0121-0026-0001', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0107-0021-0001', 'reason': 'không xác định duy nhất span đáp án sau tách'}, {'id': '0106-0046-0002', 'reason': 'không xác định duy nhất span đáp án sau tách'}, 

## Bước 4: Nạp tokenizer PhoBERT và xác nhận token ID

PhoBERT cần văn bản đã tách từ. PhoBERT-base-v2 có giới hạn 256 token, vì vậy notebook dùng MAX_LENGTH là 256. Ta cần tokenizer nhanh để lấy offset_mapping. Nếu Transformers chỉ nạp được tokenizer chậm, cell thử dùng tokenizer.json và so sánh token ID với tokenizer chuẩn. Nếu không khớp thì phải dừng, không tạo nhãn sai. [Tài liệu PhoBERT](https://huggingface.co/vinai/phobert-base-v2)


In [ ]:
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, PreTrainedTokenizerFast

CHECKPOINT = "vinai/phobert-base-v2"
MAX_LENGTH = 256
DOC_STRIDE = 64

slow_tokenizer = AutoTokenizer.from_pretrained(
    CHECKPOINT,
    use_fast=False,
)

try:
    tokenizer = AutoTokenizer.from_pretrained(
        CHECKPOINT,
        use_fast=True,
    )
except Exception:
    tokenizer = None

if tokenizer is None or not tokenizer.is_fast:
    tokenizer_file = hf_hub_download(
        repo_id=CHECKPOINT,
        filename="tokenizer.json",
    )
    tokenizer = PreTrainedTokenizerFast(
        tokenizer_file=tokenizer_file,
        bos_token=slow_tokenizer.bos_token,
        eos_token=slow_tokenizer.eos_token,
        unk_token=slow_tokenizer.unk_token,
        sep_token=slow_tokenizer.sep_token,
        cls_token=slow_tokenizer.cls_token,
        pad_token=slow_tokenizer.pad_token,
        mask_token=slow_tokenizer.mask_token,
    )

# So sánh cả câu hỏi và context để kiểm tra cách ghép cặp đầu vào.
sample_question = train_segmented[0]["question"]
sample_context = train_segmented[0]["context"]

slow_ids = slow_tokenizer(
    sample_question,
    sample_context,
)["input_ids"]
fast_ids = tokenizer(
    sample_question,
    sample_context,
)["input_ids"]

print("Tokenizer nhanh:", tokenizer.is_fast)
print("Token ID giống tokenizer gốc:", slow_ids == fast_ids)
print("MAX_LENGTH:", MAX_LENGTH, "| DOC_STRIDE:", DOC_STRIDE)

if not tokenizer.is_fast:
    raise RuntimeError("Tokenizer cần hỗ trợ offset_mapping.")
if slow_ids != fast_ids:
    raise RuntimeError(
        "Token ID không khớp tokenizer gốc; dừng trước khi tạo nhãn."
    )


## Bước 5: Token hóa, chia cửa sổ và tạo nhãn start/end

Mỗi token context có offset ký tự. Dùng offset đó để đổi span đáp án thành hai chỉ số token: start_positions và end_positions. Context dài được chia thành các cửa sổ chồng lặp. Cửa sổ không chứa đáp án và câu hỏi không đáp án được gán nhãn CLS.


In [ ]:
from datasets import Dataset

def tao_features(batch, co_metadata=False):
    encoded = tokenizer(
        batch["question"],
        batch["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = encoded.pop("overflow_to_sample_mapping")
    all_offsets = encoded.pop("offset_mapping")
    start_positions = []
    end_positions = []
    feature_ids = []
    feature_offsets = []

    for feature_index, offsets in enumerate(all_offsets):
        input_ids = encoded["input_ids"][feature_index]
        cls_index = input_ids.index(tokenizer.cls_token_id)
        sample_index = sample_mapping[feature_index]
        sequence_ids = encoded.sequence_ids(feature_index)

        answers = batch["answers"][sample_index]
        is_impossible = batch["is_impossible"][sample_index]

        if is_impossible or not answers["text"] or not answers["answer_start"]:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
        else:
            char_start = answers["answer_start"][0]
            char_end = char_start + len(answers["text"][0])

            context_indices = [
                i for i, sequence_id in enumerate(sequence_ids)
                if sequence_id == 1
            ]
            first_context = context_indices[0]
            last_context = context_indices[-1]

            # Nếu span không nằm trọn trong cửa sổ này, gán CLS.
            if (
                offsets[first_context][0] > char_start
                or offsets[last_context][1] < char_end
            ):
                start_positions.append(cls_index)
                end_positions.append(cls_index)
            else:
                token_start = first_context
                while (
                    token_start <= last_context
                    and offsets[token_start][0] <= char_start
                ):
                    token_start += 1
                start_positions.append(token_start - 1)

                token_end = last_context
                while offsets[token_end][1] >= char_end:
                    token_end -= 1
                end_positions.append(token_end + 1)

        if co_metadata:
            feature_ids.append(batch["id"][sample_index])
            feature_offsets.append([
                offset if sequence_ids[i] == 1 else None
                for i, offset in enumerate(offsets)
            ])

    encoded["start_positions"] = start_positions
    encoded["end_positions"] = end_positions

    if co_metadata:
        encoded["example_id"] = feature_ids
        encoded["offset_mapping"] = feature_offsets

    return encoded


train_dataset = Dataset.from_list(train_segmented)
dev_dataset = Dataset.from_list(dev_segmented)

train_features = train_dataset.map(
    tao_features,
    batched=True,
    remove_columns=train_dataset.column_names,
    desc="Token hóa train",
)

dev_features_with_metadata = dev_dataset.map(
    lambda batch: tao_features(batch, co_metadata=True),
    batched=True,
    remove_columns=dev_dataset.column_names,
    desc="Token hóa dev",
)

# Metadata dùng để giải mã, không đưa vào đầu vào của model.
dev_features = dev_features_with_metadata.remove_columns(
    ["example_id", "offset_mapping"]
)

print("Train mẫu gốc:", len(train_dataset))
print("Train cửa sổ token:", len(train_features))
print("Dev mẫu gốc:", len(dev_dataset))
print("Dev cửa sổ token:", len(dev_features))


## Bước 6: Chuẩn bị metrics toàn bộ, answerable và unanswerable

Dự đoán được giải mã về context đã tách từ, sau đó dấu gạch dưới đổi thành khoảng trắng để so sánh với đáp án gốc. Ngoài EM/F1 toàn bộ, notebook tính EM/F1 cho câu có đáp án và accuracy cho câu không có đáp án.


In [ ]:
import sys
import numpy as np

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.evaluation import decode_predictions, score_prediction

dev_examples_by_id = {}
for original_example, segmented_example in zip(dev_examples, dev_segmented):
    dev_examples_by_id[original_example["id"]] = {
        "context": segmented_example["context"],
        "answers": original_example["answers"],
        "is_impossible": original_example["is_impossible"],
    }

dev_feature_ids = list(dev_features_with_metadata["example_id"])
dev_feature_offsets = list(
    dev_features_with_metadata["offset_mapping"]
)


def tinh_metrics(evaluation_prediction):
    start_logits, end_logits = evaluation_prediction.predictions[:2]

    segmented_predictions = decode_predictions(
        start_logits,
        end_logits,
        dev_feature_ids,
        dev_feature_offsets,
        dev_examples_by_id,
    )

    all_em = []
    all_f1 = []
    answerable_em = []
    answerable_f1 = []
    unanswerable_correct = []

    for example_id, example in dev_examples_by_id.items():
        prediction = segmented_predictions[example_id].replace("_", " ")
        references = example["answers"].get("text", [])
        em, f1 = score_prediction(prediction, references)

        all_em.append(em)
        all_f1.append(f1)

        if example["is_impossible"]:
            unanswerable_correct.append(
                float(not prediction.strip())
            )
        else:
            answerable_em.append(em)
            answerable_f1.append(f1)

    return {
        "exact_match": 100 * float(np.mean(all_em)),
        "f1": 100 * float(np.mean(all_f1)),
        "answerable_em": 100 * float(np.mean(answerable_em)),
        "answerable_f1": 100 * float(np.mean(answerable_f1)),
        "unanswerable_accuracy": 100 * float(
            np.mean(unanswerable_correct)
        ),
    }


print("Dev mẫu gốc:", len(dev_examples_by_id))
print("Dev cửa sổ:", len(dev_feature_ids))


## Bước 7: Cấu hình fine-tune PhoBERT

Mỗi seed bắt đầu lại từ checkpoint PhoBERT gốc. Batch 4 với tích lũy gradient 8 lần tương đương batch hiệu dụng 32. Model tốt nhất được chọn bằng F1 trên dev. Chỉ chạy sau khi Bước 3 không còn lỗi ánh xạ và Bước 4 xác nhận token ID khớp.


In [ ]:
import inspect
import json
import time
import torch

from transformers import (
    AutoModelForQuestionAnswering,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)
from src.utils import save_json

if not torch.cuda.is_available():
    raise RuntimeError("Không phát hiện GPU CUDA; chưa bắt đầu huấn luyện.")

use_bf16 = torch.cuda.is_bf16_supported()
use_fp16 = not use_bf16
SEEDS = [42, 123, 1024]
RESULTS_DIR = project_root / "results" / "finetune" / "phobert"

print("GPU:", torch.cuda.get_device_name(0))
print("BF16:", use_bf16, "| FP16:", use_fp16)


def tao_training_arguments(output_dir, seed):
    arguments = {
        "output_dir": str(output_dir),
        "learning_rate": 2e-5,
        "per_device_train_batch_size": 4,
        "per_device_eval_batch_size": 8,
        "gradient_accumulation_steps": 8,
        "num_train_epochs": 3,
        "weight_decay": 0.01,
        "bf16": use_bf16,
        "fp16": use_fp16,
        "logging_strategy": "steps",
        "logging_steps": 100,
        "save_strategy": "epoch",
        "load_best_model_at_end": True,
        "metric_for_best_model": "f1",
        "greater_is_better": True,
        "save_total_limit": 2,
        "dataloader_num_workers": 0,
        "report_to": "none",
        "seed": seed,
        "data_seed": seed,
    }

    supported = inspect.signature(
        TrainingArguments.__init__
    ).parameters
    strategy_name = (
        "eval_strategy"
        if "eval_strategy" in supported
        else "evaluation_strategy"
    )
    arguments[strategy_name] = "epoch"
    arguments = {
        key: value for key, value in arguments.items()
        if key in supported
    }

    return TrainingArguments(**arguments)


## Bước 8: Huấn luyện ba seed và lưu từng lượt chạy

Vòng lặp tạo model mới cho mỗi seed, train, chọn checkpoint tốt nhất trên dev và lưu metrics cùng dự đoán. Mỗi seed có thư mục riêng nên có thể xem lại kết quả.


In [ ]:
all_seed_results = []

for seed in SEEDS:
    print("\\n" + "=" * 70)
    print("PhoBERT — seed:", seed)
    print("=" * 70)

    set_seed(seed)
    seed_dir = RESULTS_DIR / f"seed-{seed}"
    best_model_dir = seed_dir / "best_model"
    seed_dir.mkdir(parents=True, exist_ok=True)

    model = AutoModelForQuestionAnswering.from_pretrained(CHECKPOINT)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    trainer_parameters = inspect.signature(
        Trainer.__init__
    ).parameters
    tokenizer_argument = (
        {"processing_class": tokenizer}
        if "processing_class" in trainer_parameters
        else {"tokenizer": tokenizer}
    )

    trainer = Trainer(
        model=model,
        args=tao_training_arguments(seed_dir, seed),
        train_dataset=train_features,
        eval_dataset=dev_features,
        data_collator=data_collator,
        compute_metrics=tinh_metrics,
        **tokenizer_argument,
    )

    start_time = time.time()
    trainer.train()
    training_seconds = time.time() - start_time

    # Trainer đã nạp checkpoint có F1 dev tốt nhất.
    final_metrics = trainer.evaluate()
    trainer.save_model(str(best_model_dir))
    tokenizer.save_pretrained(best_model_dir)

    prediction_output = trainer.predict(dev_features)
    start_logits, end_logits = prediction_output.predictions[:2]
    segmented_predictions = decode_predictions(
        start_logits,
        end_logits,
        dev_feature_ids,
        dev_feature_offsets,
        dev_examples_by_id,
    )

    prediction_rows = []
    for example_id, example in dev_examples_by_id.items():
        prediction_rows.append({
            "id": example_id,
            "prediction_text": segmented_predictions[example_id].replace(
                "_", " "
            ),
            "ground_truths": example["answers"].get("text", []),
            "is_impossible": example["is_impossible"],
        })

    result = {
        "model": "phobert",
        "checkpoint": CHECKPOINT,
        "seed": seed,
        "max_length": MAX_LENGTH,
        "doc_stride": DOC_STRIDE,
        "exact_match": float(
            final_metrics.get("eval_exact_match", 0.0)
        ),
        "f1": float(final_metrics.get("eval_f1", 0.0)),
        "answerable_em": float(
            final_metrics.get("eval_answerable_em", 0.0)
        ),
        "answerable_f1": float(
            final_metrics.get("eval_answerable_f1", 0.0)
        ),
        "unanswerable_accuracy": float(
            final_metrics.get("eval_unanswerable_accuracy", 0.0)
        ),
        "training_seconds": round(training_seconds, 2),
        "bf16": use_bf16,
    }

    save_json(seed_dir / "metrics.json", result)
    save_json(seed_dir / "predictions.json", prediction_rows)
    all_seed_results.append(result)

    print("EM:", result["exact_match"], "| F1:", result["f1"])
    print("F1 có đáp án:", result["answerable_f1"])
    print("Accuracy không đáp án:", result["unanswerable_accuracy"])
    print("Đã lưu:", seed_dir)


## Bước 9: Tính mean +- std và lưu tổng hợp

Mean là điểm trung bình qua ba seed. Độ lệch chuẩn (std) cho biết kết quả dao động nhiều hay ít. Báo cáo seed F1 dev cao nhất riêng, đồng thời dùng mean ± std để thể hiện độ ổn định.


In [ ]:
metric_names = [
    "exact_match",
    "f1",
    "answerable_em",
    "answerable_f1",
    "unanswerable_accuracy",
]

summary = {
    "model": "phobert",
    "checkpoint": CHECKPOINT,
    "seeds": SEEDS,
    "runs": all_seed_results,
    "metrics_mean_std": {},
}

for metric_name in metric_names:
    values = [row[metric_name] for row in all_seed_results]
    summary["metrics_mean_std"][metric_name] = {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
    }

best_run = max(all_seed_results, key=lambda row: row["f1"])
summary["best_seed"] = best_run["seed"]

save_json(RESULTS_DIR / "summary.json", summary)

print("Seed có F1 dev cao nhất:", summary["best_seed"])
for metric_name, values in summary["metrics_mean_std"].items():
    print(
        metric_name,
        f"{values['mean']:.2f} ± {values['std']:.2f}",
    )
print("Đã lưu:", RESULTS_DIR / "summary.json")


## Bước 10: Kiểm tra và ghi kết quả vào báo cáo

Kết quả nằm trong results/finetune/phobert/. Mỗi thư mục seed có best_model, metrics.json và predictions.json; summary.json chứa điểm từng seed, mean ± std và seed tốt nhất.

Trước khi đưa vào báo cáo, xác nhận đủ ba seed, dev vẫn có 3.814 mẫu, metrics không bị thiếu và ghi rõ PhoBERT dùng Underthesea, MAX_LENGTH 256, DOC_STRIDE 64, batch hiệu dụng 32, 3 epoch. Chỉ dùng test.json cho lần đánh giá cuối cùng sau khi đã chốt mô hình bằng dev.
